<a href="https://colab.research.google.com/github/Ch01pan/DataScience_230401010199_Arsina/blob/main/Pertemuan12_%5BARSINA%5D_%5B230401010199%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hands On 12**

Nama : Arsina

NIM : 230401010199

Kelas : IF 405

# **1. Generate & Eksplorasi Dataset Transaksi**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Daftar produk
np.random.seed(42)

produk = [
    "Roti", "Selai", "Susu", "Sereal", "Telur",
    "Keju", "Kopi", "Gula", "Teh", "Mentega"
]

# Membuat 50 transaksi
transaksi = []

for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(
        list(np.random.choice(produk, n_item, replace=False))
    )

# Menambahkan pola: Roti sering dibeli bersama Selai
for i in range(20):
    if "Roti" in transaksi[i] and "Selai" not in transaksi[i]:
        transaksi[i].append("Selai")

# Menampilkan informasi dataset
print("Contoh transaksi:")
for i, t in enumerate(transaksi[:3], start=1):
    print(f"Transaksi {i}: {t}")

print("\nJumlah transaksi:", len(transaksi))

Contoh transaksi:
Transaksi 1: [np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai']
Transaksi 2: [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')]
Transaksi 3: [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]

Jumlah transaksi: 50


# **2. One-Hot Encoding Transaksi**

In [2]:
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

# Melakukan one-hot encoding pada data transaksi
te = TransactionEncoder()

te_ary = te.fit(transaksi).transform(transaksi)

# Membuat DataFrame hasil encoding
df = pd.DataFrame(
    te_ary,
    columns=te.columns_
)

# Menampilkan 5 data pertama
print("Hasil One-Hot Encoding:")
print(df.head())

Hasil One-Hot Encoding:
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


# **3. Cari Frequent Itemset dengan Apriori**

In [3]:
import warnings
warnings.filterwarnings("ignore")

import os

os.environ["PYTHONWARNINGS"] = "ignore"

import warnings

warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="jupyter_client"
)


In [12]:
import warnings
warnings.filterwarnings('ignore')
from mlxtend.frequent_patterns import apriori

# Mencoba beberapa nilai minimum support
for ms in [0.05, 0.10, 0.20]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f"min_support = {ms}: {len(freq)} itemset ditemukan")

# Menggunakan nilai minimum support yang dianggap paling sesuai
freq_items = apriori(df, min_support=0.10, use_colnames=True)

# Mengurutkan berdasarkan nilai support tertinggi
freq_items = freq_items.sort_values("support", ascending=False)

# Menampilkan 10 itemset teratas
print("\n10 Frequent Itemset Teratas:")
print(freq_items.head(10))

min_support = 0.05: 74 itemset ditemukan
min_support = 0.1: 44 itemset ditemukan
min_support = 0.2: 13 itemset ditemukan

10 Frequent Itemset Teratas:
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


# **4. Bentuk & Saring Aturan Asosiasi**

In [11]:
import warnings
warnings.filterwarnings('ignore')
from mlxtend.frequent_patterns import association_rules

# Membentuk association rules berdasarkan confidence
rules = association_rules(
    freq_items,
    metric="confidence",
    min_threshold=0.5
)

# Menyaring aturan dengan nilai lift > 1
rules = rules[rules["lift"] > 1]

# Mengurutkan berdasarkan nilai lift tertinggi
rules = rules.sort_values(
    by="lift",
    ascending=False
)

# Menampilkan 10 aturan asosiasi teratas
print("10 Association Rules Teratas:")
print(
    rules[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(10)
)

10 Association Rules Teratas:
         antecedents consequents  support  confidence      lift
9        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
15  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
12      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
13     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
8      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
11     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
14   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


# **5. Rekomender Sederhana dengan Content-Based Filtering**

In [10]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Membuat katalog produk
katalog = pd.DataFrame({
    "produk": produk,
    "kategori": [
        "Bakery", "Bakery", "Dairy", "Bakery", "Dairy",
        "Dairy", "Minuman", "Bumbu", "Minuman", "Dairy"
    ]
})

# One-Hot Encoding kategori
fitur = pd.get_dummies(katalog["kategori"])

# Menghitung cosine similarity
sim_matrix = cosine_similarity(fitur)

# Fungsi rekomendasi produk serupa
def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog["produk"] == nama_produk][0]

    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)

    # Menghapus produk yang sama dari hasil rekomendasi
    skor = [s for s in skor if s[0] != idx][:top_n]

    return katalog.iloc[[i for i, _ in skor]]["produk"].tolist()

# Contoh rekomendasi
print("Mirip dengan Roti:", rekomendasi_serupa("Roti"))



Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


# **6. Bandingkan Kedua Pendekatan**

In [13]:

# Produk yang akan dicari rekomendasinya
produk_target = "Roti"

# Mencari aturan asosiasi yang memiliki antecedent mengandung produk target
rules_terkait = rules[
    rules["antecedents"].apply(lambda x: produk_target in x)
]

# Menampilkan rekomendasi dari Association Rules
print("Rekomendasi dari Association Rules:")
print(
    rules_terkait[
        ["consequents", "lift"]
    ].head()
)

# Menampilkan rekomendasi dari Content-Based Filtering
print("\nRekomendasi dari Content-Based:")
print(rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115

Rekomendasi dari Content-Based:
['Selai', 'Sereal', 'Susu']


# **Kesimpulan**

Hands On 12,  algoritma Association Rule Mining (Apriori) dan Content-Based Filtering sukses diterapkan pada 50 data transaksi sintetis yang telah melalui proses one-hot encoding. Evaluasi menggunakan Apriori dengan min_support 0,10 menghasilkan 44 frequent itemset, di mana salah satu aturan asosiasi terkuatnya adalah Roti → Selai. Sementara itu, pendekatan Content-Based Filtering menggunakan cosine similarity pada produk Roti juga memunculkan rekomendasi Selai (bersama Sereal dan Susu) berdasarkan kemiripan kategori. Hasil konsisten dari kedua metode ini menegaskan fungsinya masing-masing: Association Rules handal membaca pola histori transaksi, sedangkan Content-Based efektif mencocokkan karakteristik produk. Kombinasi keduanya akan menghasilkan sistem rekomendasi pelanggan yang jauh lebih komprehensif.